In [1]:
import pandas as pd
import numpy as np
import re

from scipy.interpolate import griddata
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("../data/raw/dataset.csv")

ce_cols = [c for c in df.columns if c.endswith("CE")]
pe_cols = [c for c in df.columns if c.endswith("PE")]

In [3]:
def get_strike(col):
    return int(
        re.search(
            r'JAN26(\d+)(CE|PE)$',
            col
        ).group(1)
    )

In [4]:
def build_surface(df, cols):

    records = []

    for t in range(len(df)):

        for col in cols:

            records.append({
                "time_idx": t,
                "strike": get_strike(col),
                "iv": df.loc[t,col]
            })

    return pd.DataFrame(records)

In [8]:
def surface_fill(surface_df):

    known = surface_df[
        surface_df["iv"].notna()
    ]

    missing = surface_df[
        surface_df["iv"].isna()
    ]

    surface_df = surface_df.copy()

    # Linear interpolation
    linear_values = griddata(
        points=known[
            ["time_idx","strike"]
        ].values,

        values=known["iv"].values,

        xi=missing[
            ["time_idx","strike"]
        ].values,

        method="linear"
    )

    # Nearest-neighbor fallback
    nearest_values = griddata(
        points=known[
            ["time_idx","strike"]
        ].values,

        values=known["iv"].values,

        xi=missing[
            ["time_idx","strike"]
        ].values,

        method="nearest"
    )

    final_values = np.where(
        np.isnan(linear_values),
        nearest_values,
        linear_values
    )

    surface_df.loc[
        missing.index,
        "iv"
    ] = final_values

    return surface_df

In [9]:
def surface_validation(cols,
                       hide_fraction=0.20,
                       seed=42):

    np.random.seed(seed)

    temp = df.copy()

    known_positions = []

    for col in cols:

        valid_rows = temp.index[
            temp[col].notna()
        ]

        for r in valid_rows:

            known_positions.append(
                (r,col)
            )

    n_hide = int(
        len(known_positions)
        * hide_fraction
    )

    selected = np.random.choice(
        len(known_positions),
        n_hide,
        replace=False
    )

    hidden = []

    for idx in selected:

        r,col = known_positions[idx]

        true_val = temp.loc[r,col]

        hidden.append(
            (r,col,true_val)
        )

        temp.loc[r,col] = np.nan

    surface = build_surface(
        temp,
        cols
    )

    surface = surface_fill(
        surface
    )

    pred_lookup = {}

    for _,row in surface.iterrows():

        pred_lookup[
            (
                row["time_idx"],
                row["strike"]
            )
        ] = row["iv"]

    y_true = []
    y_pred = []

    for r,col,true_val in hidden:

        strike = get_strike(col)

        pred = pred_lookup[
            (r,strike)
        ]

        y_true.append(true_val)
        y_pred.append(pred)

    print(
        "NaNs in prediction:",
        np.isnan(y_pred).sum()
    )

    return mean_squared_error(
        y_true,
        y_pred
    )

In [10]:
results = []

for seed in [1,2,3,4,5]:

    ce_mse = surface_validation(
        ce_cols,
        seed=seed
    )

    pe_mse = surface_validation(
        pe_cols,
        seed=seed
    )

    results.append({
        "seed":seed,
        "ce_surface":ce_mse,
        "pe_surface":pe_mse
    })

pd.DataFrame(results)

NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0


,seed,ce_surface,pe_surface
0,1,0.000189,0.005653
1,2,0.000306,0.003146
2,3,0.000408,0.004960
3,4,0.000430,0.000366
4,5,0.000311,0.000381


In [11]:
option_cols = ce_cols + pe_cols

gap_sizes = []

for _, row in df[option_cols].isna().iterrows():
    count = 0

    for val in row:
        if val:
            count += 1
        else:
            if count > 0:
                gap_sizes.append(count)
            count = 0

    if count > 0:
        gap_sizes.append(count)

from collections import Counter

print(Counter(gap_sizes))

Counter({1: 3547, 2: 697, 3: 123, 4: 30, 5: 6})


In [12]:
option_cols = ce_cols + pe_cols

single_gap = 0
multi_gap = 0

for _, row in df[option_cols].isna().iterrows():

    count = 0

    for val in row:

        if val:
            count += 1

        else:

            if count == 1:
                single_gap += 1

            elif count > 1:
                multi_gap += count

            count = 0

    if count == 1:
        single_gap += 1

    elif count > 1:
        multi_gap += count

print("single_gap_cells =", single_gap)
print("multi_gap_cells =", multi_gap)

single_gap_cells = 3547
multi_gap_cells = 1913


# ATM-aware

In [23]:
from scipy.interpolate import interp1d
import numpy as np

def atm_aware_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    spot = row["underlying_price"]

    atm_idx = np.argmin(
        np.abs(strikes - spot)
    )

    result = values.copy()

    # LEFT SIDE
    left_strikes = strikes[:atm_idx+1]
    left_values = values[:atm_idx+1]

    mask = ~np.isnan(left_values)

    if mask.sum() >= 2:

        f = interp1d(
            left_strikes[mask],
            left_values[mask],
            kind="linear",
            fill_value="extrapolate",
            bounds_error=False
        )

        missing = np.isnan(left_values)

        result[:atm_idx+1][missing] = f(
            left_strikes[missing]
        )

    # RIGHT SIDE
    right_strikes = strikes[atm_idx:]
    right_values = values[atm_idx:]

    mask = ~np.isnan(right_values)

    if mask.sum() >= 2:

        f = interp1d(
            right_strikes[mask],
            right_values[mask],
            kind="linear",
            fill_value="extrapolate",
            bounds_error=False
        )

        missing = np.isnan(right_values)

        result[atm_idx:][missing] = f(
            right_strikes[missing]
        )
    
    # Fallback for anything ATM-aware missed
    mask = ~np.isnan(values)

    if np.isnan(result).any() and mask.sum() >= 2:

        f = interp1d(
            strikes[mask],
            values[mask],
            kind="linear",
            fill_value="extrapolate",
            bounds_error=False
        )

        missing = np.isnan(result)

        result[missing] = f(
            strikes[missing]
    )

    return result

In [24]:
import re
import numpy as np

def get_strike(col):
    return int(
        re.search(
            r'JAN26(\d+)(CE|PE)$',
            col
        ).group(1)
    )

ce_strikes = np.array(
    [get_strike(c) for c in ce_cols]
)

pe_strikes = np.array(
    [get_strike(c) for c in pe_cols]
)

print(ce_strikes)
print(pe_strikes)

[25200 25300 25400 25500 25600 25700 25800 25900 26000 26100 26200 26300
 26400 26500]
[23800 23900 24000 24100 24200 24300 24400 24500 24600 24700 24800 24900
 25000 25100]


In [26]:
from sklearn.metrics import mean_squared_error

def evaluate_atm(seed=42):

    np.random.seed(seed)

    temp = df.copy()

    known_locations = []

    for col in ce_cols + pe_cols:

        valid_rows = temp.index[
            temp[col].notna()
        ].tolist()

        for r in valid_rows:
            known_locations.append((r,col))

    known_locations = np.array(
        known_locations,
        dtype=object
    )

    n_hide = int(
        len(known_locations) * 0.20
    )

    selected_idx = np.random.choice(
        len(known_locations),
        n_hide,
        replace=False
    )

    hidden = []

    for idx in selected_idx:

        row,col = known_locations[idx]

        true_value = temp.loc[row,col]

        hidden.append(
            (row,col,true_value)
        )

        temp.loc[row,col] = np.nan

    for idx in temp.index:

        temp.loc[idx,ce_cols] = atm_aware_row(
            temp.loc[idx],
            ce_cols,
            ce_strikes
        )

        temp.loc[idx,pe_cols] = atm_aware_row(
            temp.loc[idx],
            pe_cols,
            pe_strikes
        )

    y_true = []
    y_pred = []

    for row,col,true_value in hidden:

        y_true.append(true_value)
        y_pred.append(temp.loc[row,col])

    print("NaNs in y_pred:", np.isnan(y_pred).sum())

    return mean_squared_error(
        y_true,
        y_pred
    )

In [27]:
results = []

for seed in [1,2,3,4,5]:

    mse = evaluate_atm(seed)

    results.append({
        "seed":seed,
        "atm_mse":mse
    })

results_df = pd.DataFrame(results)

results_df

NaNs in y_pred: 0
NaNs in y_pred: 0
NaNs in y_pred: 0
NaNs in y_pred: 0
NaNs in y_pred: 0


,seed,atm_mse
0,1,0.000067
1,2,0.000096
2,3,0.000137
3,4,0.000051
4,5,0.000111


In [28]:
results_df.mean(
    numeric_only=True
)

seed       3.000000
atm_mse    0.000092
dtype: float64